# Thực hành Hàng đợi và BPMN — Khoa học dịch vụ

**Hai bài thực hành:** (1) đối chiếu Erlang C với mô phỏng M/M/s; (2) chuyển quy trình BPMN sân bay thành hai trạm hàng đợi.

**Chú ý:** tất cả số liệu là *giả định để học tập*, không phải dữ liệu vận hành của sân bay. Các chỉ số mô phỏng có dao động và cần đối chiếu với giả định mô hình.

## 0. Chuẩn bị

Mở trong Google Colab hoặc Jupyter Notebook. Ở Colab, chạy ô cài đặt để có thư viện `simpy`; trong một môi trường Jupyter đã cài SimPy có thể bỏ qua lệnh này.

- Trang dự án SimPy: https://simpy.readthedocs.io/
- Công cụ mở tệp BPMN: https://demo.bpmn.io/new
- Mô hình đi kèm: `airport_checkin_queue_demo.bpmn`

In [ ]:
%pip -q install simpy numpy pandas matplotlib

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy

pd.set_option('display.precision', 3)
SEEDS = range(2026, 2034)

## 1. Mô hình M/M/s — kiểm chứng công thức Erlang C

Giả thiết: khách đến theo Poisson với `lambda_rate=18 lượt/giờ`, thời gian phục vụ mũ `mu_rate=12 lượt/giờ/quầy`. Mỗi khách vào **một hàng chung**, đến trước được phục vụ trước. Thử với 2 và 3 quầy.

Công thức (với \(a=\lambda/\mu,\ \rho=a/s<1\)):

\[
p_0=\left[\sum_{n=0}^{s-1}\frac{a^n}{n!}+\frac{a^s}{s!(1-\rho)}\right]^{-1},\quad
P(\\text{chờ})=\frac{a^s}{s!(1-\rho)}p_0,\quad
W_q=\frac{P(\\text{chờ})}{s\mu-\lambda}.
\]

Đơn vị đầu ra `Wq` theo phút, nên nhân 60 khi tham số đang đo theo giờ.

In [ ]:
def erlang_c(lam, mu, servers):
    a = lam / mu
    rho = a / servers
    if rho >= 1:
        return {'rho': rho, 'p_wait': np.nan, 'wq_minutes': np.nan}
    tail = (a**servers / math.factorial(servers)) / (1 - rho)
    p0 = 1 / (sum(a**k / math.factorial(k) for k in range(servers)) + tail)
    p_wait = tail * p0
    wq_minutes = 60 * p_wait / (servers * mu - lam)
    return {'rho': rho, 'p_wait': p_wait, 'wq_minutes': wq_minutes}

pd.DataFrame([{'s': s, **erlang_c(18, 12, s)} for s in [1, 2, 3, 4]])

### Mô phỏng M/M/s bằng SimPy

Thời gian mô phỏng tính theo **giờ**. Bỏ `warmup_hours` giờ đầu nhằm giảm ảnh hưởng hệ thống khởi đầu rỗng; cho hệ thống phục vụ hết các khách đã đến trước mốc kết thúc. Chạy 8 hạt giống và báo số trung bình qua các lần chạy.

In [ ]:
def sim_mms(lam=18, mu=12, servers=2, horizon_hours=1100, warmup_hours=150, seed=2026):
    rng_arrivals = np.random.default_rng(seed)
    rng_services = np.random.default_rng(seed + 10_000)
    env = simpy.Environment()
    counter = simpy.Resource(env, capacity=servers)
    waits = []

    def customer(arrival, service_duration):
        with counter.request() as req:
            yield req
            wq = env.now - arrival
            yield env.timeout(service_duration)
        if arrival >= warmup_hours:
            waits.append(wq * 60)

    def arrivals():
        while True:
            gap = rng_arrivals.exponential(1 / lam)
            if env.now + gap >= horizon_hours:
                break
            yield env.timeout(gap)
            duration = rng_services.exponential(1 / mu)
            env.process(customer(env.now, duration))

    env.process(arrivals())
    env.run()  # phục vụ hết khách đã đến trước horizon_hours
    a = np.asarray(waits)
    return dict(s=servers, seed=seed, n=len(a), wq_minutes=float(a.mean()),
                p_wait=float(np.mean(a > 1e-10)), p90_minutes=float(np.percentile(a, 90)))

experiments = pd.DataFrame([sim_mms(servers=s, seed=seed) for s in [2, 3] for seed in SEEDS])
summary = experiments.groupby('s').agg(
    n_runs=('seed', 'count'),
    Wq_mo_phong=('wq_minutes', 'mean'),
    Wq_do_lech_giua_runs=('wq_minutes','std'),
    P_cho=('p_wait','mean'),
    Wq_phan_vi_90=('p90_minutes','mean')).reset_index()
summary['Wq_ErlangC'] = summary['s'].map(lambda s: erlang_c(18, 12, s)['wq_minutes'])
print(summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for _, r in summary.iterrows():
    s = int(r.s)
    ax.scatter(s, r.Wq_mo_phong, s=85, label='SimPy' if s == 2 else None)
    ax.scatter(s, r.Wq_ErlangC, s=90, marker='x', label='Erlang C' if s == 2 else None)
ax.set(xlabel='Số quầy', ylabel='Wq trung bình (phút)', xticks=[2, 3])
ax.grid(alpha=.2); ax.legend(); plt.show()

**Thảo luận:** thời gian chờ trung bình của nhiều lần mô phỏng cần gần kết quả Erlang C, nhưng một lần mô phỏng hữu hạn sẽ không khớp tuyệt đối. Nếu bỏ khởi động mà Wq lệch mạnh, hãy tăng thời gian mô phỏng và số lần lặp.

## 2. Từ BPMN sang mạng hàng đợi hai công đoạn

Mở `airport_checkin_queue_demo.bpmn` trên [bpmn.io](https://demo.bpmn.io/new) để xem sơ đồ nghiệp vụ: **Khách đến → Kiểm tra giấy tờ → (Có hành lý?) → Ký gửi → Hoàn tất**.

Giả định hai trạm độc lập về năng lực: 
- **Kiểm tra giấy tờ:** λ = 18 khách/giờ, \(s_1=2\), \(\mu_1=12\) khách/giờ/nhân viên.
- **Ký gửi hành lý:** 60% hành khách sử dụng, \(s_2=2\), \(\mu_2=10\) khách/giờ/nhân viên.

Với mỗi hành khách, thời gian chờ tại trạm đầu tiên và (nếu có) trạm thứ hai được ghi lại; tổng thời gian trong quy trình bao gồm cả phục vụ ở từng trạm. Tệp BPMN chỉ mô tả *luồng* — code SimPy dưới đây gắn thời gian và nguồn lực.

In [ ]:
def sim_airport(s_docs=2, s_baggage=2, bag_share=.60, lam=18,
                mu_docs=12, mu_bag=10, horizon_hours=1200,
                warmup_hours=150, seed=2026):
    env = simpy.Environment()
    docs = simpy.Resource(env, capacity=s_docs)
    baggage = simpy.Resource(env, capacity=s_baggage)
    rng_arr = np.random.default_rng(seed)
    rng_docs = np.random.default_rng(seed + 10_000)
    rng_bag = np.random.default_rng(seed + 20_000)
    rng_route = np.random.default_rng(seed + 30_000)
    rows = []

    def traveler(arrival, docs_time, has_bag, bag_time):
        with docs.request() as req:
            yield req
            w_docs = env.now - arrival
            yield env.timeout(docs_time)
        w_bag = 0.0
        if has_bag:
            arrived_bag = env.now
            with baggage.request() as req:
                yield req
                w_bag = env.now - arrived_bag
                yield env.timeout(bag_time)
        if arrival >= warmup_hours:
            rows.append({'wait_docs': 60*w_docs,
                         'wait_bag': 60*w_bag,
                         'has_bag': has_bag,
                         'total_time': 60*(env.now - arrival)})

    def arrivals():
        while True:
            gap = rng_arr.exponential(1/lam)
            if env.now + gap >= horizon_hours: break
            yield env.timeout(gap)
            docs_time = rng_docs.exponential(1/mu_docs)
            bag_time = rng_bag.exponential(1/mu_bag)
            has_bag = rng_route.random() < bag_share
            env.process(traveler(env.now, docs_time, has_bag, bag_time))

    env.process(arrivals())
    env.run()
    df = pd.DataFrame(rows)
    return {'seed':seed, 's_docs':s_docs, 's_bag':s_baggage,
            'N':len(df), 'Wq_docs':df.wait_docs.mean(),
            'Wq_bag_per_user':df.wait_bag.mean(),
            'Wq_bag_given_bag':df.loc[df.has_bag,'wait_bag'].mean(),
            'W_total':df.total_time.mean(),
            'P_cho_docs':(df.wait_docs>1e-10).mean(),
            'P90_total':df.total_time.quantile(.9)}

compare=pd.DataFrame([sim_airport(s_docs=s,seed=seed) for s in [2,3] for seed in SEEDS])
res=compare.groupby('s_docs').agg({'Wq_docs':'mean','Wq_bag_given_bag':'mean','W_total':'mean',
                                   'P_cho_docs':'mean','P90_total':'mean'}).round(2)
print(res)

## 3. Bài tập thay tham số

1. Tăng tỷ lệ có hành lý từ `bag_share=.60` lên `.80`. Trạm nào có nguy cơ thành nút nghẽn?
2. Giữ `bag_share=.60` nhưng giảm số nhân viên ký gửi từ 2 xuống 1. Kiểm tra điều kiện ổn định \(\rho_2=\lambda_2/(s_2\mu_2)\) trước khi chạy. Nếu \(\rho_2\geq 1\), không dùng công thức trạng thái ổn định.
3. Báo **Wq tại từng trạm, W toàn quy trình, P(chờ), phân vị 90** và một nhận xét ngắn về hiệu quả bổ sung nhân viên.

**Lưu ý phương pháp:** thời gian chờ tại trạm ký gửi trong bảng có cả *trung bình trên mọi hành khách* (người không gửi hành lý nhận 0) và *trung bình có điều kiện trên người ký gửi*. Không nhầm hai mẫu đo này.